In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# ==========================================
# PART 03: FINAL MAP DATASET
# ==========================================

PROJECT_DIR = Path.cwd()

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
TABLE_DIR = PROJECT_DIR / "output" / "tables"

MAP_FILE = PROCESSED_DIR / "part3_map_ready.csv"
ANNUAL_FILE = PROCESSED_DIR / "london_underground_clean.csv"
CLUSTER_FILE = TABLE_DIR / "part2_cluster_assignments.csv"

OUTPUT_FILE = TABLE_DIR / "part3_final_map.csv"

# ==========================================
# 1. Load existing files
# ==========================================

map_df = pd.read_csv(MAP_FILE)
annual = pd.read_csv(ANNUAL_FILE)
clusters = pd.read_csv(CLUSTER_FILE)

print("Map data:", map_df.shape)
print("Annual data:", annual.shape)
print("Cluster data:", clusters.shape)

# ==========================================
# 2. Prepare station identifiers
# ==========================================

map_df["Station_ID"] = pd.to_numeric(
    map_df["Station_ID"], errors="coerce"
).astype("Int64")

annual["MNLC"] = pd.to_numeric(
    annual["MNLC"], errors="coerce"
).astype("Int64")

annual = annual[annual["Mode"] == "LU"].copy()

# ==========================================
# 3. Calculate 2025 recovery by day type
# ==========================================

day_columns = {
    "MonThu": ("MTT_Entry", "MTT_Exit"),
    "Friday": ("Friday_Entry", "Friday_Exit"),
    "Saturday": ("Saturday_Entry", "Saturday_Exit"),
    "Sunday": ("Sunday_Entry", "Sunday_Exit")
}

recovery_frames = []

for day, (entry_col, exit_col) in day_columns.items():

    temp = annual[
        ["Year", "MNLC", entry_col, exit_col]
    ].copy()

    temp["Passenger_Flow"] = (
        temp[entry_col] + temp[exit_col]
    )

    pivot = temp.pivot(
        index="MNLC",
        columns="Year",
        values="Passenger_Flow"
    )

    result = pd.DataFrame({
        "Station_ID": pivot.index,
        f"Recovery_{day}": (
            pivot[2025] / pivot[2019] * 100
        )
    })

    recovery_frames.append(result)

# Merge the four day types
day_recovery = recovery_frames[0]

for frame in recovery_frames[1:]:
    day_recovery = day_recovery.merge(
        frame,
        on="Station_ID",
        how="outer",
        validate="one_to_one"
    )

# ==========================================
# 4. Merge recovery into map data
# ==========================================

map_df = map_df.merge(
    day_recovery,
    on="Station_ID",
    how="left",
    validate="one_to_one"
)

# Create recovery groups for each day type
for day in day_columns:

    map_df[f"Group_{day}"] = pd.cut(
        map_df[f"Recovery_{day}"],
        bins=[-np.inf, 90, 100, np.inf],
        labels=[
            "Below baseline",
            "Near baseline",
            "Above baseline"
        ],
        right=False
    ).astype("string")

# ==========================================
# 5. Add 2019 cluster assignments
# ==========================================

# Use 2019 as the fixed baseline classification
cluster_2019 = clusters[
    clusters["Year"] == 2019
][["ASC", "Cluster"]].copy()

# Get the MASC-to-MNLC station cross-reference
station_lookup = (
    annual[annual["Year"] == 2019]
    [["MNLC", "MASC"]]
    .drop_duplicates()
)

station_lookup["MASC"] = (
    station_lookup["MASC"].astype(str).str.strip()
)

cluster_2019["ASC"] = (
    cluster_2019["ASC"].astype(str).str.strip()
)

cluster_lookup = station_lookup.merge(
    cluster_2019,
    left_on="MASC",
    right_on="ASC",
    how="left",
    validate="one_to_one"
)

map_df = map_df.merge(
    cluster_lookup[["MNLC", "Cluster"]],
    left_on="Station_ID",
    right_on="MNLC",
    how="left",
    validate="one_to_one"
)

map_df = map_df.drop(columns=["MNLC"], errors="ignore")

# ==========================================
# 6. Map quality flags and Elizabeth line
# ==========================================

elizabeth_interchanges = {
    "bond street",
    "tottenham court road",
    "farringdon",
    "liverpool street",
    "whitechapel",
    "paddington",
    "paddington tfl",
    "moorgate",
    "ealing broadway",
    "stratford",
    "heathrow terminals 1 2 3",
    "heathrow terminals 2 3",
    "heathrow terminal 4",
    "heathrow terminal 5"
}

def normalise_name(name):
    return (
        str(name).lower()
        .replace("&", " ")
        .replace(",", " ")
        .replace("'", "")
        .strip()
    )

map_df["Elizabeth_Line_Interchange"] = (
    map_df["Station"]
    .apply(normalise_name)
    .isin(elizabeth_interchanges)
)

# Keep the stations on the map, but flag incomparable data
map_df["Comparison_Status"] = "Comparable"

map_df.loc[
    map_df["Station"].isin(["Richmond", "Wimbledon"]),
    "Comparison_Status"
] = "Not comparable — different data coverage"

map_df.loc[
    map_df["Station"].str.contains("Waterloo", case=False, na=False),
    "Comparison_Status"
] = "Low confidence — estimated 2019 baseline"

# Exclude incomparable stations from recovery rankings
map_df["Include_in_Ranking"] = (
    map_df["Comparison_Status"] == "Comparable"
)

# Weekend display values: cap at 150, preserve original values
for day in ["Saturday", "Sunday"]:
    map_df[f"Recovery_{day}_Display"] = (
        map_df[f"Recovery_{day}"].clip(upper=150)
    )

# District / Circle baseline share
district_circle = map_df["LINES"].fillna("").str.contains(
    r"\b(District|Circle)\b",
    case=False,
    regex=True
)

print("\nDistrict/Circle stations:")
print(f"{district_circle.sum()} / {len(map_df)}")
print(f"Baseline share: {district_circle.mean() * 100:.1f}%")

print("\nWeekend map note:")
print("Some weekend baseline values may be anomalous.")



# ==========================================
# 7. Export final dataset
# ==========================================

map_df.to_csv(OUTPUT_FILE, index=False)

print("\n========== FINAL AUDIT ==========")

print("Output shape:", map_df.shape)
print("Unique stations:", map_df["Station_ID"].nunique())

print("\nMissing recovery values:")
recovery_cols = [
    f"Recovery_{day}" for day in day_columns
]
print(map_df[recovery_cols].isna().sum())

print("\nCluster distribution:")
print(map_df["Cluster"].value_counts(dropna=False))

print("\nElizabeth line interchange stations:")
print(
    map_df.loc[
        map_df["Elizabeth_Line_Interchange"],
        "Station"
    ].tolist()
)

print("\nSaved to:")
print(OUTPUT_FILE)


Map data: (267, 18)
Annual data: (807, 16)
Cluster data: (798, 3)

District/Circle stations:
74 / 267
Baseline share: 27.7%

Weekend map note:
Some weekend baseline values may be anomalous.

========== FINAL AUDIT ==========
Output shape: (267, 32)
Unique stations: 267

Missing recovery values:
Recovery_MonThu      0
Recovery_Friday      0
Recovery_Saturday    0
Recovery_Sunday      0
dtype: int64

Cluster distribution:
Cluster
0.0    150
1.0    116
NaN      1
Name: count, dtype: int64

Elizabeth line interchange stations:
['Farringdon', 'Tottenham Court Road', 'Whitechapel', 'Paddington TfL', 'Bond Street', 'Moorgate', 'Ealing Broadway', 'Stratford']

Saved to:
/Users/ovo/Desktop/Grad/Internship/Data_for_Humanity_Internship/do-starter-playground/Model/output/tables/part3_final_map.csv


/var/folders/6z/2rysjv1j7vn0tfk3620xdrq00000gn/T/ipykernel_8984/3423898786.py:223: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  district_circle = map_df["LINES"].fillna("").str.contains(


In [2]:

# ==========================================
# EXTRA AUDIT: WEEKEND, COVERAGE AND SOURCE
# ==========================================

print("\n========== 1. WEEKEND RECOVERY BY LINE ==========")

# Stations with weekend recovery above 100%
sat_above = map_df["Recovery_Saturday"] > 100
sun_above = map_df["Recovery_Sunday"] > 100

weekend_any = sat_above | sun_above
weekend_both = sat_above & sun_above

# Identify District / Circle stations
district_circle = map_df["LINES"].fillna("").str.contains(
    r"\b(District|Circle)\b",
    case=False,
    regex=True
)

def report_group(label, mask):
    total = int(mask.sum())
    dc_count = int((mask & district_circle).sum())
    percentage = dc_count / total * 100 if total else 0

    print(f"\n{label}")
    print("Stations above 100%:", total)
    print("District/Circle stations:", dc_count)
    print(f"Share: {percentage:.1f}%")

    print(
        map_df.loc[
            mask,
            ["Station", "LINES", "Recovery_Saturday", "Recovery_Sunday"]
        ]
        .sort_values("Recovery_Saturday", ascending=False)
        .to_string(index=False)
    )

report_group("Saturday > 100%", sat_above)
report_group("Sunday > 100%", sun_above)
report_group("Either weekend day > 100%", weekend_any)
report_group("Both Saturday and Sunday > 100%", weekend_both)

print("\nDistrict/Circle stations in the full map:")
print(int(district_circle.sum()))

# ==========================================
# 2. Coverage and Source consistency
# ==========================================

print("\n========== 2. COVERAGE AND SOURCE ==========")

stations_to_check = ["Richmond", "Wimbledon"]

coverage_check = annual[
    (annual["Station"].str.strip().str.lower().isin(
        [s.lower() for s in stations_to_check]
    ))
    & (annual["Year"].isin([2019, 2025]))
][
    ["Station", "Year", "MNLC", "MASC", "Coverage", "Source"]
].sort_values(["Station", "Year"])

print(coverage_check.to_string(index=False))

for station in stations_to_check:
    rows = coverage_check[
        coverage_check["Station"].str.strip().str.lower()
        == station.lower()
    ]

    print(f"\n{station}:")
    if set(rows["Year"]) != {2019, 2025}:
        print("Missing one of the two years.")
        continue

    for column in ["Coverage", "Source"]:
        values = rows.set_index("Year")[column]
        same = values.loc[2019] == values.loc[2025]
        print(
            f"{column} consistent:",
            same,
            f"(2019: {values.loc[2019]} | 2025: {values.loc[2025]})"
        )



========== 1. WEEKEND RECOVERY BY LINE ==========

Saturday > 100%
Stations above 100%: 87
District/Circle stations: 49
Share: 56.3%
                 Station                                                                    LINES  Recovery_Saturday  Recovery_Sunday
              Farringdon                                 Metropolitan, Circle, Hammersmith & City         329.406805       365.029790
    Tottenham Court Road                                                        Central, Northern         183.179043       171.029091
             Whitechapel                                             District, Hammersmith & City         180.380335       188.061700
          Mill Hill East                                                                 Northern         176.793817       184.992103
               Wimbledon                                                                 District         163.750656       135.566097
     Harrow & Wealdstone                                      

/var/folders/6z/2rysjv1j7vn0tfk3620xdrq00000gn/T/ipykernel_8984/4177925832.py:15: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  district_circle = map_df["LINES"].fillna("").str.contains(


In [3]:

import pandas as pd
from pathlib import Path

# Read existing annual dataset
PROJECT_DIR = Path.cwd()

annual = pd.read_csv(
    PROJECT_DIR / "data" / "processed" / "london_underground_clean.csv"
)

# Exclude the 11 stations
exclude_stations = [
    "Richmond",
    "Wimbledon",
    "Farringdon",
    "Tottenham Court Road",
    "Whitechapel",
    "Paddington TfL",
    "Bond Street",
    "Liverpool Street",
    "Moorgate",
    "Stratford",
    "Ealing Broadway"
]

# London Underground only
annual = annual[annual["Mode"] == "LU"].copy()

# Remove the 11 stations from all years
annual = annual[
    ~annual["Station"].str.strip().isin(exclude_stations)
].copy()

# Calculate network-wide passenger flow
day_columns = {
    "Mon-Thu": ("MTT_Entry", "MTT_Exit"),
    "Friday": ("Friday_Entry", "Friday_Exit"),
    "Saturday": ("Saturday_Entry", "Saturday_Exit"),
    "Sunday": ("Sunday_Entry", "Sunday_Exit")
}

totals = {}

for day, (entry, exit) in day_columns.items():
    totals[day] = (
        annual.groupby("Year")[entry].sum()
        + annual.groupby("Year")[exit].sum()
    )

# Recovery relative to 2019
recovery = pd.DataFrame(totals).T

recovery = recovery.div(recovery[2019], axis=0) * 100

print("Network-wide recovery after excluding 11 stations:")
display(recovery.round(1))


Network-wide recovery after excluding 11 stations:


Year,2019,2022,2025
Mon-Thu,100.0,74.2,78.3
Friday,100.0,70.5,71.6
Saturday,100.0,89.1,91.0
Sunday,100.0,86.5,91.4
